# Pump Health Monitoring - Analysis Notebook

This notebook demonstrates the pump health monitoring system including:
- Data loading and exploration
- Feature engineering
- Model training and evaluation
- Predictions and visualizations

In [ ]:
import sys
from pathlib import Path

# Add src to path
sys.path.append(str(Path.cwd().parent / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_ingestion.data_loader import PumpDataLoader
from feature_engineering.feature_builder import PumpFeatureBuilder
from rul_prediction.predictor import RULPredictor
from config import SENSOR_COLUMNS, TARGET_COLUMN, MODEL_CONFIG

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

## 1. Data Loading and Exploration

In [ ]:
# Load data
data_loader = PumpDataLoader()
df = data_loader.generate_synthetic_data(n_samples=10000, n_pumps=5)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

In [ ]:
# Basic statistics
df.describe()

In [ ]:
# Visualize sensor data for one pump
pump_0 = df[df['pump_id'] == 0].head(500)

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('Sensor Data Over Time - Pump 0', fontsize=16)

sensors = ['flow_rate', 'pressure_in', 'pressure_out', 'temperature', 
           'vibration', 'power_consumption', 'rpm', 'rul']

for idx, sensor in enumerate(sensors):
    ax = axes[idx // 3, idx % 3]
    ax.plot(pump_0['timestamp'], pump_0[sensor])
    ax.set_title(sensor.replace('_', ' ').title())
    ax.set_xlabel('Time')
    ax.set_ylabel('Value')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 2. Feature Engineering

In [ ]:
# Build features
feature_builder = PumpFeatureBuilder(window_size=10)
df_features = feature_builder.build_all_features(df, SENSOR_COLUMNS)

print(f"Original features: {len(df.columns)}")
print(f"Engineered features: {len(df_features.columns)}")
print(f"\nNew feature count: {len(df_features.columns) - len(df.columns)}")

In [ ]:
# Correlation matrix for key features
key_features = ['flow_rate', 'pressure_differential', 'temperature', 'vibration', 
                'efficiency', 'degradation_index', 'rul']

plt.figure(figsize=(10, 8))
sns.heatmap(df_features[key_features].corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Matrix - Key Features')
plt.tight_layout()
plt.show()

## 3. Model Training

In [ ]:
# Train model
predictor = RULPredictor(model_type='xgboost')

X_train, X_test, y_train, y_test = predictor.prepare_data(
    df_features,
    target_column=TARGET_COLUMN,
    test_size=MODEL_CONFIG['test_size'],
    random_state=MODEL_CONFIG['random_state']
)

predictor.train(X_train, y_train, xgb_params=MODEL_CONFIG['xgboost_params'])

## 4. Model Evaluation

In [ ]:
# Evaluate model
metrics = predictor.evaluate(X_test, y_test)

print("\nModel Performance:")
print(f"RMSE: {metrics['rmse']:.2f} hours")
print(f"MAE: {metrics['mae']:.2f} hours")
print(f"R² Score: {metrics['r2']:.3f}")

In [ ]:
# Predictions vs Actuals
y_pred = predictor.model.predict(X_test)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual RUL (hours)')
plt.ylabel('Predicted RUL (hours)')
plt.title('Predicted vs Actual RUL')

plt.subplot(1, 2, 2)
residuals = y_test - y_pred
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--', lw=2)
plt.xlabel('Predicted RUL (hours)')
plt.ylabel('Residuals')
plt.title('Residual Plot')

plt.tight_layout()
plt.show()

## 5. Feature Importance

In [ ]:
# Feature importance
importance_df = predictor.get_feature_importance(top_n=15)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'], importance_df['importance'])
plt.xlabel('Importance')
plt.title('Top 15 Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Health Index Calculation

In [ ]:
# Calculate health indices
health_indices = predictor.predict_health_index(y_pred, max_rul=1000)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(health_indices, bins=50, edgecolor='black')
plt.xlabel('Health Index')
plt.ylabel('Frequency')
plt.title('Distribution of Health Indices')

plt.subplot(1, 2, 2)
plt.scatter(y_pred, health_indices, alpha=0.5)
plt.xlabel('Predicted RUL (hours)')
plt.ylabel('Health Index')
plt.title('Health Index vs Predicted RUL')

plt.tight_layout()
plt.show()

## 7. Example Predictions

In [ ]:
# Make predictions on a specific pump
pump_id = 0
df_pump = df_features[df_features['pump_id'] == pump_id].tail(100)

feature_cols = [col for col in df_features.columns 
                if col not in [TARGET_COLUMN, 'timestamp', 'pump_id']]
X_pump = df_pump[feature_cols]

rul_predictions = predictor.predict(X_pump.values)
health_predictions = predictor.predict_health_index(rul_predictions, max_rul=1000)

# Plot predictions
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(df_pump['timestamp'].values, rul_predictions, label='Predicted RUL', marker='o')
plt.plot(df_pump['timestamp'].values, df_pump[TARGET_COLUMN].values, 
         label='Actual RUL', marker='x')
plt.xlabel('Time')
plt.ylabel('RUL (hours)')
plt.title(f'RUL Prediction - Pump {pump_id}')
plt.legend()
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
plt.plot(df_pump['timestamp'].values, health_predictions, marker='o', color='green')
plt.axhline(y=0.7, color='orange', linestyle='--', label='Good Threshold')
plt.axhline(y=0.5, color='red', linestyle='--', label='Fair Threshold')
plt.xlabel('Time')
plt.ylabel('Health Index')
plt.title(f'Health Index - Pump {pump_id}')
plt.legend()
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
print(f"Average predicted RUL: {rul_predictions.mean():.2f} hours")
print(f"Average health index: {health_predictions.mean():.3f}")
print(f"\nPumps requiring attention (health < 0.7): {(health_predictions < 0.7).sum()}")
print(f"Pumps in critical condition (health < 0.5): {(health_predictions < 0.5).sum()}")